In [0]:
%run "../../commons/commons_imports"

In [0]:
df_municipio_bronze = read(
    base_path=BRONZE_PATH,
    table_name=TS_MUNICIPIO,
    recursive_by_year=False,
    format = "delta",
)

In [0]:
df_municipio_silver = (

    df_municipio_bronze

    # =====================================================
    # Região
    # =====================================================

    .withColumn(
        "REGIAO",
        when(col("SG_UF").isin("AC","AP","AM","PA","RO","RR","TO"), "Norte")
        .when(col("SG_UF").isin("AL","BA","CE","MA","PB","PE","PI","RN","SE"), "Nordeste")
        .when(col("SG_UF").isin("DF","GO","MT","MS"), "Centro-Oeste")
        .when(col("SG_UF").isin("ES","MG","RJ","SP"), "Sudeste")
        .when(col("SG_UF").isin("PR","RS","SC"), "Sul")
    )

    # =====================================================
    # Município + UF
    # =====================================================

    .withColumn(
        "NO_MUNICIPIO_UF",
        concat_ws(
            " - ",
            col("NO_MUNICIPIO"),
            col("SG_UF")
        )
    )

    # =====================================================
    # Faixa da Média LP
    # =====================================================

    .withColumn(
        "FAIXA_MEDIA_LP",
        when(col("VL_MEDIA_LP").isNull(), "Não avaliado")
        .when(col("VL_MEDIA_LP") < 700, "<700")
        .when(col("VL_MEDIA_LP") < 743, "700-742")
        .when(col("VL_MEDIA_LP") < 800, "743-799")
        .otherwise("800+")
    )

    # =====================================================
    # Faixa de Alfabetização
    # =====================================================

    .withColumn(
        "FAIXA_ALFABETIZACAO",
        when(col("PC_ALUNO_ALFABETIZADO").isNull(), "Não informado")
        .when(col("PC_ALUNO_ALFABETIZADO") < 40, "Muito Baixa")
        .when(col("PC_ALUNO_ALFABETIZADO") < 60, "Baixa")
        .when(col("PC_ALUNO_ALFABETIZADO") < 80, "Boa")
        .otherwise("Excelente")
    )

    # =====================================================
    # Possui distribuição por níveis
    # =====================================================

    .withColumn(
        "IN_POSSUI_DISTRIBUICAO_NIVEIS",
        when(
            col("PC_ALUNO_NIVEL_0_LP").isNotNull(),
            1
        ).otherwise(0)
    )

    # =====================================================
    # Auditoria
    # =====================================================

    .withColumn(
        "ANO_CARGA",
        year(col("TS_PROCESSAMENTO"))
    )

    .withColumn(
        "MES_CARGA",
        month(col("TS_PROCESSAMENTO"))
    )

)

In [0]:
df_municipio_silver_selected = df_municipio_silver.select(

    # ============================================
    # Chave Técnica
    # ============================================
    "SK_MUNICIPIO",

    # ============================================
    # Chaves de Negócio
    # ============================================
    "NU_ANO_AVALIACAO",
    "ANO_REFERENCIA",

    "CO_UF",
    "SG_UF",
    "REGIAO",

    "CO_MUNICIPIO",

    "NO_MUNICIPIO",
    "NO_MUNICIPIO_UF",

    # ============================================
    # Informações da Rede
    # ============================================
    "TP_SERIE",
    "DS_SERIE",

    "ID_TIPO_REDE",
    "DS_TIPO_REDE",

    # ============================================
    # Indicadores
    # ============================================
    "PC_ALUNO_ALFABETIZADO",
    "FAIXA_ALFABETIZACAO",

    "VL_MEDIA_LP",
    "FAIXA_MEDIA_LP",

    # ============================================
    # Distribuição por Níveis
    # ============================================
    "PC_ALUNO_NIVEL_0_LP",
    "PC_ALUNO_NIVEL_1_LP",
    "PC_ALUNO_NIVEL_2_LP",
    "PC_ALUNO_NIVEL_3_LP",
    "PC_ALUNO_NIVEL_4_LP",
    "PC_ALUNO_NIVEL_5_LP",
    "PC_ALUNO_NIVEL_6_LP",
    "PC_ALUNO_NIVEL_7_LP",
    "PC_ALUNO_NIVEL_8_LP",

    # ============================================
    # Auditoria
    # ============================================
    "ANO_CARGA",
    "MES_CARGA",

    "DT_PROCESSAMENTO",
    "TS_PROCESSAMENTO"
)

In [0]:
validate_primary_key(df_municipio_silver_selected, "SK_MUNICIPIO")

validate_not_null(
    df_municipio_silver_selected,
    [
        "SK_MUNICIPIO"
    ]
)

validate_years(df_municipio_silver_selected)

In [0]:
write_delta(
    df=df_municipio_silver_selected,
    base_path=SILVER_PATH,
    table_name=TS_MUNICIPIO,
    merge_keys=["SK_MUNICIPIO"]
)